In [1]:
"""
Run FlamMap command line interface version
1) baseline FlamMap (pre-treatment)
2) treated FlamMap (post-treatment)

Maxwell.Cook@colostate.edu
"""

import os, sys
from pathlib import Path
from tqdm.notebook import tqdm

# Handle weird GDAL dependencies once
prefix = Path(sys.prefix)
os.environ["PROJ_LIB"] = str(prefix / "Library" / "share" / "proj")
os.environ["GDAL_DATA"] = str(prefix / "Library" / "share" / "gdal")

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from code.Python.archive.__functions import *  # imports all custom functions

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")

print("Success !")

Project directory set to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper
Success !


In [ ]:
# --- Load the baseline + treated fuelscapes
baseline_fp = list_files(projdir, "fuelscape_baseline.tif", recursive=True)[0]

treated_dir = os.path.join(projdir, "data/spatial/fuelscape/NCFC/treated/landscape")
treated_fps = sorted([str(p) for p in Path(treated_dir).glob("*.tif")])

# load into a unified dictionary
fuelscapes = {"Baseline": baseline_fp}
for fp in treated_fps:
    name = Path(fp).stem # file name --> treatment scenario
    fuelscapes[name] = fp

print(f"Fuelscapes loaded:\n{fuelscapes.keys()}")

In [ ]:
# --- Specify FlamMap.exe
flammap_exe = list_files(projdir, "TestFlamMap.exe", recursive=True)[0]
print(f"FlamMap.exe exists? {os.path.exists(flammap_exe)}")

# --- Weather scenarios table
scenario_df = pd.read_csv(list_files(os.path.join(projdir,'data/spatial/flammap'),
                                     "fire_scenarios.csv", recursive=True)[0])
scenario_df = scenario_df[~scenario_df['Scenario'].isin(['pct75','pct100'])]
print(f"Scenarios:\n\t{scenario_df['Scenario'].unique()}")

# create an output directory
output_dir = Path(os.path.join(projdir, r'data/spatial/flammap/outputs'))
os.makedirs(output_dir, exist_ok=True)
print(f"Saving FlamMap outputs to: {output_dir}")

In [ ]:
# keep only the unique weather scenarios once (pct25..pct100)
# (adjust this if your CSV is already weather-only)
weather = scenario_df.drop(columns=["LCP", "LCPType"], errors="ignore")
weather = weather.drop_duplicates(subset=["Scenario"]).reset_index(drop=True)

# cartesian product with fuelscapes
rows = []
for _, w in weather.iterrows():
    for lcp_type, lcp_fp in fuelscapes.items():
        r = w.copy()
        r["LCPType"] = lcp_type          # "baseline" or "Complete_Hand"
        r["LCP"] = Path(lcp_fp)                # full path to tif
        rows.append(r)

scenarios = pd.DataFrame(rows)
print("Total runs:", len(scenarios))
print(scenarios[["Scenario","LCPType","LCP"]].head())

In [ ]:
scenarios.columns

In [ ]:
outputs = ['FLAMELENGTH', 'CROWNSTATE', 'SPREADRATE']
scenarios['Outputs'] = ','.join(outputs)
scenarios['Outputs'].unique()

In [ ]:
scenarios['LCP'].unique()

In [ ]:
def stack_rasters(in_dir, tag=None, out_dir=None, cleanup=True):
    """

    :param in_dir:
    :param tag:
    :param out_dir:
    :param cleanup:
    :return:
    """

    # situate the directories
    in_dir = Path(in_dir)
    if out_dir is None:
        out_dir = in_dir
    else:
        out_dir = Path(out_dir)

    # tag is the fuelscape we're using (e.g., baseline)
    if tag is None:
        tag = in_dir.name.split("_")[1].upper()  # assumes naming like scenario_tag

    # file_prefix is the scenario (e.g., 'pct25')
    file_prefix = in_dir.name.split("_")[0].upper()

    # find the raster files for the current scenario or raise an error
    tifs = list(Path(in_dir).glob('*.tif'))
    if not tifs:
        raise FileNotFoundError(f"No tifs found in {in_dir}")
    tifs.sort()

    # Load and stack rasters
    bands = []
    band_names = []
    for tif in tifs:
        # get the band naming convention
        band_name = str(tif.stem).split("_")[1]
        band_names.append(band_name)  # use filename stem as band name
        # open the band, name it
        with rxr.open_rasterio(tif) as da:
            da = da.squeeze()  # single band
            da.attrs["long_name"] = f"{file_prefix}_{band_name}"
            bands.append(da)  # append the band

    # stack the individual bands, assign band names
    stack = xr.concat(bands, dim=pd.Index(band_names, name="band"))

    # save the tif file out
    out_fp = out_dir / f"{file_prefix}_{tag.upper()}.tif"
    stack.rio.to_raster(out_fp, compress="deflate")
    print(f"Stacked {len(tifs)} rasters to: {out_fp}")

    del stack, bands
    gc.collect() # clean up

    # cleanup the directories if specified (default True)
    if cleanup:
        for tif in tifs:
            tif.unlink()
            aux_fp = tif.with_suffix(".tif.aux")
            if aux_fp.exists():
                aux_fp.unlink() # delete the FlamMap tifs and aux files


def cli_flammap_scenarios(
        fm_exe, lcp_fp, fm_params, output_directory, n_process=1,
        tag=None, stack_out=False, cleanup=False
):
    """
    Initiates a FlamMap CLI run for a given fire weather scenario and landscape file.

    :param fm_exe: the executable for TestFlamMap
    :param lcp_fp: landscape/fuelscape file
    :param fm_params: scenario table defining the model inputs
    :param output_directory: Where to save the outputs
    :param n_process: number of cores to use (I think?)
    :param tag: tag to identify this run (optional)
    :param stack_out: Stack rasters into output folder
    :param cleanup: Clean up the output folder
    :return: All requested FlamMap outputs for each fire scenario.
    """

    # get the fuelscape name as the tag
    if tag is None:
        tag = fm_params['LCPType']

    # initiate a FlamMap input file and command file
    input_file = output_directory / "FlamMap.input"
    command_file = output_directory / "FMcommand.txt"

    # write contents to the input file
    with open(input_file, "w") as f:
        # write the header
        f.write("ShortTerm-Inputs-File-Version-1\n\n")

        # specify fuel moisture and other methods
        f.write("FUEL_MOISTURES_DATA: 1\n")
        f.write(f"0 {fm_params['FM_1hr']} {fm_params['FM_10hr']} "
                f"{fm_params['FM_100hr']} {fm_params['FM_herb']} {fm_params['FM_woody']}\n")
        f.write("FOLIAR_MOISTURE_CONTENT: 100\n")
        # specify the crown fire methodology
        f.write(f"CROWN_FIRE_METHOD: {str(fm_params['CROWN_FIRE_METHOD'])}\n")
        # processors (could experiment with higher values for greater computation?)
        f.write(f"NUMBER_PROCESSORS: {n_process}\n") # default number of processors to 1
        # specify wind information (speed and direction)
        f.write(f"WIND_SPEED: {fm_params['WIND_SPEED']}\n") # wind speed
        f.write(f"WIND_DIRECTION: {fm_params['WIND_DIRECTION']}\n") # note that -1 is uphill and -2 is downhill
        # WindNinja settings:
        if fm_params["GRIDDED_WINDS_GENERATE"] == "Yes":
            # turn on WindNinja
            f.write("GRIDDED_WINDS_GENERATE: Yes\n")
            # set the resolution
            f.write(f"GRIDDED_WINDS_RESOLUTION: {fm_params['GRIDDED_WINDS_RESOLUTION']}\n")

        # specify the desired outputs
        outputs = [o.strip() for o in str(fm_params["Outputs"]).split(",") if o.strip()]
        for output_type in outputs:
            f.write(f"{output_type}: 1\n")

    # open and edit the command file (tells FlamMap what to run)
    with open(command_file, "w") as f:
        f.write(f'{str(lcp_fp)} {str(input_file.name)} . 2\n')

    # Run FlamMap scenario
    # log the outputs
    with open(output_directory / "flammap_run.log", "w") as log_file:
        subprocess.run(
            [str(fm_exe), str(command_file.name)],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
            cwd=output_directory
        )

    # stack the rasters into a multi-band output, if specified
    if stack_out is True:
        stack_rasters(
            in_dir=output_directory,
            cleanup=cleanup
        )

In [ ]:
# --- Run the fuelscape X scenario through FlamMap
output_root = Path(os.path.join(projdir, r"data\spatial\flammap\outputs"))
output_root.mkdir(parents=True, exist_ok=True)

# sort the scenarios table
scenarios = scenarios.sort_values(["LCPType", "Scenario"]).reset_index(drop=True)

n_runs = len(scenarios)
for _, row in tqdm(scenarios.iterrows(), total=n_runs, desc="Running FlamMap Scenarios"):

    scenario_id = row["Scenario"]        # e.g., pct97 (weather scenario)
    lcp_type = row["LCPType"]         # e.g., baseline, Complete_Hand
    lcp_fp = row["LCP"]             # full path to the landscape file

    # create a folder for this treatment scenario
    run_dir = output_root / lcp_type #  (e.g., outputs/baseline, outputs/complete_hand, etc.)
    run_dir.mkdir(parents=True, exist_ok=True)

    tqdm.write(f"\nProcessing [{scenario_id}] for fuelscape=[{lcp_type}]")

    # run the CLI batch
    cli_flammap_scenarios(
        fm_exe=flammap_exe,
        lcp_fp=lcp_fp,
        fm_params=row,
        output_directory=run_dir,
        n_process=6,
        stack_out=True,
        cleanup=False
    )